# Calculating Power Law Distribution of Allan Factor

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import plotly.express as px
import plotly.graph_objects as go

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'allan_var-multiling.tsv'

PARAMETERS_PATH = 'av-params-multiling.csv'

vis_path = DATA_PATH.replace('.csv', '.html')

In [3]:
subcorpora_col = 'corpus'

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

#### Loading Results

In [5]:
results = pd.read_csv(PARAMETERS_PATH)

In [6]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0
0,callfriend-eng-n-callfriend-eng-n-4175-10,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.620213,-7.079351
1,callfriend-eng-n-callfriend-eng-n-4175-10,False,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.060613,-8.374164
2,callfriend-eng-n-callfriend-eng-n-4175-100,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.614538,-10.620731
3,callfriend-eng-n-callfriend-eng-n-4175-100,False,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.258000,-9.561489
4,callfriend-eng-n-callfriend-eng-n-4175-1000,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.140355,-8.255489


#### Other checks

In [7]:
results.isna().sum()

x_turn_id            0
null                 0
self                 0
conversation_id      0
speaker              0
slope              541
b0                 549
dtype: int64

In [8]:
results.isna().mean()

x_turn_id          0.000000
null               0.000000
self               0.000000
conversation_id    0.000000
speaker            0.000000
slope              0.000775
b0                 0.000787
dtype: float64

#### Collecting instances where slope is bad

In [9]:
(results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean()

(np.int64(5517), np.float64(0.007906752771018032))

In [10]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0
0,callfriend-eng-n-callfriend-eng-n-4175-10,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.620213,-7.079351
1,callfriend-eng-n-callfriend-eng-n-4175-10,False,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.060613,-8.374164
2,callfriend-eng-n-callfriend-eng-n-4175-100,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.614538,-10.620731
3,callfriend-eng-n-callfriend-eng-n-4175-100,False,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.258000,-9.561489
4,callfriend-eng-n-callfriend-eng-n-4175-1000,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.140355,-8.255489


In [11]:
sel = results['slope'].isna() | (results['slope'].abs() == np.inf) 
sel.sum(), sel.mean(), (~sel).sum()

(np.int64(6058), np.float64(0.008682093218565749), np.int64(691700))

In [12]:
results['self'].value_counts()

self
True     379625
False    318133
Name: count, dtype: int64

In [15]:
results['x_turn_id'].loc[~sel].nunique()

385587

In [16]:
is_null = results['x_turn_id'].str.contains('null-')
is_null.sum()

np.int64(57827)

Adding Corpus names back in . . .

In [17]:
results[subcorpora_col] = [lang(x) for x in tqdm(results['conversation_id'])]

  0%|          | 0/697758 [00:00<?, ?it/s]

In [18]:
results[subcorpora_col].value_counts()

corpus
spa    215687
eng    117343
cro    101237
zho     90481
fra     64135
ko      58536
deu     45896
fin      3936
yid       507
Name: count, dtype: int64

In [19]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0,corpus
0,callfriend-eng-n-callfriend-eng-n-4175-10,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.620213,-7.079351,eng
1,callfriend-eng-n-callfriend-eng-n-4175-10,False,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.060613,-8.374164,eng
2,callfriend-eng-n-callfriend-eng-n-4175-100,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.614538,-10.620731,eng
3,callfriend-eng-n-callfriend-eng-n-4175-100,False,True,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,0.258000,-9.561489,eng
4,callfriend-eng-n-callfriend-eng-n-4175-1000,False,False,callfriend-eng-n-callfriend-eng-n-4175,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.140355,-8.255489,eng


## Aggregate corpus statistis

#### Self values

What is the aggregate slope for the whole corpus with regards to self? Let's find that out here! The following tests what the slope is for "self" in the data for both the real data and the null-distribution data separately. Both are reported on in the final paper story.

##### Real data

In [20]:
sel_ = ~sel & (results['self']) & (~is_null)
sel_.sum()

np.int64(347985)

In [22]:
mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))
mu, SE

(np.float64(-0.11175473196810914), np.float64(0.0014943010894522325))

t-test for good measure.

In [23]:
mu / SE

np.float64(-74.78729203702528)

##### Null data

In [24]:
sel_ = (~sel) & (results['self']) & (is_null)

mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))

T = mu / SE

In [25]:
sel_.sum(), mu, SE, T

(np.int64(28909),
 np.float64(0.001033597586340728),
 np.float64(0.005023999409084769),
 np.float64(0.2057320278485105))

#### Other values

What is the aggregate slope for the whole corpus with regards to others? Let's find that out here! The following tests what the slope is for "other" in the data for both the real data and the null-distribution data separately. Both are reported on in the final paper story.

##### Real data

In [26]:
sel_ = ~sel & (~results['self']) & (~is_null)
sel_.sum()

np.int64(285890)

In [27]:
mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))
mu, SE

(np.float64(-0.06045385856837153), np.float64(0.0017546659812154356))

t-test for good measure.

In [28]:
mu / SE

np.float64(-34.45320033303198)

##### Null data

In [29]:
sel_ = ~sel & (~results['self']) & (is_null)

mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))

T = mu / SE

In [30]:
sel_.sum(), mu, SE, T

(np.int64(28916),
 np.float64(-0.00316387939765035),
 np.float64(0.004962370802430356),
 np.float64(-0.6375741603390092))

## Removing null data for other statistics

In [27]:
print(results.shape)
results = results.loc[~is_null]
results.shape

(697616, 7)


(639931, 7)

## individual corpora statistics

In [28]:
subcorpus_results = []

In [29]:
for corpus in tqdm(results[subcorpora_col].unique()):
    
    res = {'corpus': corpus}
    
    sub = results.loc[
        results[subcorpora_col].isin([corpus])
        & (~sel) & (~is_null)
    ]
    
    # self vals
    sel__ = sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['self_alpha'] = mu
    res['self_se'] = se
    res['self_n'] = n
    res['self_t'] = 't({})={}'.format(n-1, mu/se)
    
    # self vals
    sel__ = ~sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['other_alpha'] = mu
    res['other_se'] = se
    res['other_n'] = n
    res['other_t'] = 't({})={}'.format(n-1, mu/se)
    
    # save outputs
    subcorpus_results += [res]

subcorpus_results = pd.DataFrame(subcorpus_results)    

  0%|          | 0/9 [00:00<?, ?it/s]

In [30]:
subcorpus_results.head(n=100)

,corpus,self_alpha,self_se,self_n,self_t,other_alpha,other_se,other_n,other_t
0,eng,-0.119274,0.003882,53937,t(53936)=-30.727354507776212,-0.040306,0.004257,50699,t(50698)=-9.468837808341418
1,fra,-0.103747,0.004362,32534,t(32533)=-23.78330114595253,-0.043004,0.005063,26141,t(26140)=-8.49300897458941
2,ko,-0.141988,0.005849,26719,t(26718)=-24.277415084733246,-0.072866,0.006433,26293,t(26292)=-11.32642940634177
3,spa,-0.107855,0.002524,111407,t(111406)=-42.72476386945069,-0.065722,0.003135,85778,t(85777)=-20.963256629115843
4,zho,-0.111434,0.003680,54919,t(54918)=-30.28013013819235,-0.077504,0.005573,28571,t(28570)=-13.90828379933875
5,deu,-0.100091,0.006741,20870,t(20869)=-14.847894341512045,-0.069202,0.007492,20597,t(20596)=-9.23636099044241
6,cro,-0.106940,0.004275,45331,t(45330)=-25.016874346815012,-0.056963,0.003935,45967,t(45966)=-14.475941547711361
7,fin,-0.094879,0.035293,1802,t(1801)=-2.688285629600637,-0.178555,0.036614,1811,t(1810)=-4.87668117138846
8,yid,-0.093025,0.054380,466,t(465)=-1.7106583573973204,0.586779,0.500324,33,t(32)=1.1727963969112225


In [31]:
# reporting['Var'] = reporting.index.values
# 
# with open(lme_results_name.replace('.csv', '2.txt'), 'w') as f:
#     txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
# 
#     txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
#     f.write(txt)
#     f.close()

## Individual differences between speakers

### Self
In effect, we are testing whether or not there are individual differences in the degree to which speakers drift from their own ideas from turn to turn.

To test this, we first calculate the mean difference of the exponential rate of decay ($\alpha$) for all speakers compared to all other speakers. We then estimate the variance in rate of decay as a direct measurement of individual differences.

In [32]:
sel_ = ~sel & (results['self']) #& (~is_null)
sel_.sum()

np.int64(347985)

In [33]:
df = results[['speaker', 'slope']].loc[sel_].groupby('speaker').agg('mean').reset_index()

In [34]:
df.head()

,speaker,slope
0,callfriend-eng-n-callfriend-eng-n-4175-F1,0.444924
1,callfriend-eng-n-callfriend-eng-n-4175-F2,-0.230333
2,callfriend-eng-n-callfriend-eng-n-4175-M1,-0.127861
3,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.123341
4,callfriend-eng-n-callfriend-eng-n-4504-EPH,-0.177271


In [35]:
delta_xy = np.abs(df['slope'].values - df['slope'].values.reshape(-1,1))
delta_xy = np.tril(delta_xy)
N = (delta_xy != 0)
N.sum()

np.int64(4048435)

In [36]:
mu = delta_xy.sum() / N.sum()
mu

np.float64(0.3107094100431431)

In [37]:
var = (((delta_xy - mu) * N )**2).sum()/(N.sum() - 1)
np.sqrt(var)

np.float64(0.5913008289527867)

In [38]:
SE = np.sqrt(var/(N.sum()-1))
SE

np.float64(0.0002938765657827637)

In [39]:
mu / np.sqrt(var/N.sum())

np.float64(1057.2787509938594)

### Other

Are there individual differences in the degree to which individual speakers influence their conversation partners? We replicate the same procedure in the "self" approach above on data for exponential decay rate ($\alpha$) for the influence of an utterance on the "other".

In [40]:
sel_ = ~sel & (~results['self']) #& (~is_null)
sel_.sum()

np.int64(285890)

In [41]:
df = results[['speaker', 'slope']].loc[sel_].groupby('speaker').agg('mean').reset_index()

In [42]:
df.head()

,speaker,slope
0,callfriend-eng-n-callfriend-eng-n-4175-F1,-0.095012
1,callfriend-eng-n-callfriend-eng-n-4175-F2,0.520012
2,callfriend-eng-n-callfriend-eng-n-4175-M1,0.106305
3,callfriend-eng-n-callfriend-eng-n-4175-M2,-0.087777
4,callfriend-eng-n-callfriend-eng-n-4504-EPH,-0.258364


In [43]:
delta_xy = np.abs(df['slope'].values - df['slope'].values.reshape(-1,1))
delta_xy = np.tril(delta_xy)
N = (delta_xy != 0)
N.sum()

np.int64(3904615)

In [44]:
mu = delta_xy.sum() / N.sum()
mu

np.float64(0.27548352236670803)

In [45]:
var = (((delta_xy - mu) * N )**2).sum()/(N.sum() - 1)
np.sqrt(var)

np.float64(0.38214617263664846)

In [46]:
SE = np.sqrt(var/(N.sum()-1))
SE

np.float64(0.00019339287099915805)

In [47]:
mu / np.sqrt(var/N.sum())

np.float64(1424.4762809512788)